# The four decision examples of TRL, one after the other

TRL's `examples/decisions/` holds four training examples for decision models, each trained with the experimental
`DecisionTrainer`. This notebook runs each of them briefly on one machine and then asks the trained model one
request, to show that every example trains, evaluates, saves and answers with the pinned forks:

1. **CLM**: projection heads on a frozen SmolLM2-135M encoder, on typed decisions;
2. **Jeva**: a Jeva head with LoRA on a Qwen3.5-4B encoder, on BANKING77;
3. **JEV**: a full fine-tune of a JEV-9B checkpoint, on requests built from typed decisions;
4. **Clef**: a cosine head trained with its KLM-1.6 backbone, on typed decisions.

Run it on a Linux machine with an NVIDIA GPU. With `QUICK = True` (the default) each stage is a short run on a
sample; its numbers only show that the stage works. Without a GPU, CLM and Clef run a few steps on the CPU and Jeva
and JEV are skipped, with the reason.

The notebook makes its own virtual environment inside the run folder, so the Python packages already on the machine
are neither used nor changed. This kernel needs nothing but the standard library: everything that touches torch runs
in fresh processes of that environment. Each training run is the example's own script with its own configuration
file, started with `accelerate launch`; what this notebook changes is given on the command line.

## Settings

This is the only cell to edit.

In [ ]:
from pathlib import Path

# Where the code comes from: your forks, pinned to the commits this notebook was written against
TRANSFORMERS_SOURCE = "https://github.com/ericlewis/transformers/archive/c6261d1e1ea691c900e70bb2d3f9de91a8e32644.tar.gz"
TRL_SOURCE = "https://github.com/ericlewis/trl/archive/bc673dcd0060a05426012a309993475d91f19a09.tar.gz"
FLA_SOURCE = "https://github.com/fla-org/flash-linear-attention/archive/9c102739e348968753d86bb729c57dd6efa1c413.tar.gz"
CLEF_ATTENTION = "kernels-community/flash-attn2@291664546de978ab2bb730554252e5e2c1099e4f"  # the Clef backbone's attention on CUDA; None: default

# Where it runs
ENVIRONMENT = "own"  # "own": a virtual environment in the run folder, with its own torch. "kernel": this kernel's Python
TORCH = "torch==2.14.1"  # for "own"; the build for this machine's GPU and driver is picked automatically
INSTALL = True  # False: use the environment as it is

# What runs
RUN = ["clm", "jeva", "jev", "clef"]  # the stages to run; a stage left out of the list is skipped
QUICK = True  # True: a short run of each stage on a sample. False: each example's full recipe
NUM_GPUS = None  # None: every visible GPU (CUDA_VISIBLE_DEVICES is honoured)
SEED = 3407

# The data of CLM, JEV and Clef (Jeva trains on BANKING77, which its example loads itself)
DATASET = "LocalLLaMA/typed-decisions"  # rows with `state`, `questions` and `gold`
DATASET_CONFIG = "all"

# The model of each stage
CLM_ENCODER = "HuggingFaceTB/SmolLM2-135M"
JEVA_ENCODER = "Qwen/Qwen3.5-4B"
JEV_MODEL = "autotrust/JEV-9B"
JEV_OPTIM = "sgd"  # the JEV fine-tune took about 53 GB with SGD (measured); AdamW needs about 90 (estimated, not run)
JEV_MIN_FREE_GB = 60  # JEV is skipped when less memory than this (GB) is available to the GPU; about 95 for AdamW
CLEF_MODEL = "MinimaLabs/KLM-1.6"
CLEF_MODEL_REVISION = "8fabc98abe194f7450cf19ed743caa7e85a4e5b0"

WORK_DIR = Path("decision_examples_run").resolve()

## Environment

With `ENVIRONMENT = "own"` this makes a virtual environment in the run folder with [uv](https://docs.astral.sh/uv/),
installs torch for this machine's GPU, then the released packages for their dependencies, then the forks and FLA
(KLM's kernels, for the Clef stage) over them. Nothing outside the run folder is changed. The first build downloads a
few GB of CUDA libraries, so it can take several minutes; the output shows its progress.

`ENVIRONMENT = "kernel"` installs into this kernel's Python instead and keeps its torch. Either way the environment is
then checked: it must import the four models and hold the pinned forks.

In [ ]:
import importlib.util
import json
import os
import shutil
import subprocess
import sys

WORK_DIR.mkdir(parents=True, exist_ok=True)
PACKAGES = ["transformers", "trl", "accelerate>=1.15.0", "datasets", "peft", "kernels", "einops", "pyyaml"]
FORKS = [TRANSFORMERS_SOURCE, TRL_SOURCE, FLA_SOURCE]


def run_environment():
    """The environment of a run: this one's, without anything left by another distributed job."""
    distributed = ("RANK", "LOCAL_RANK", "WORLD_SIZE", "LOCAL_WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT")
    environment = {k: v for k, v in os.environ.items() if not k.startswith("ACCELERATE_") and k not in distributed}
    environment["TOKENIZERS_PARALLELISM"] = "false"
    environment["PYTHONWARNINGS"] = "ignore"
    environment.setdefault("TRITON_CACHE_DIR", str(WORK_DIR / "cache" / "triton"))  # compiled kernels, kept between runs
    return environment


def stream(command):
    """Run a command in a fresh process, print its output as it comes, and stop the notebook if it fails."""
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=WORK_DIR, env=run_environment(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"The command stopped with exit code {process.returncode}; its last lines are above")


def in_python(code, python=None):
    """Run a line of Python outside the kernel; returns (worked, what it printed, the end of its errors)."""
    done = subprocess.run([str(python or PY), "-c", code], capture_output=True, text=True, env=run_environment())
    return done.returncode == 0, done.stdout.strip(), done.stderr.strip()[-1500:]


if ENVIRONMENT == "own":
    PY = WORK_DIR / "env" / "bin" / "python"
    if INSTALL or not PY.exists():
        if shutil.which("uv") is None and importlib.util.find_spec("uv") is None:
            stream([sys.executable, "-m", "pip", "install", "-q", "uv"])
        uv = [shutil.which("uv")] if shutil.which("uv") else [sys.executable, "-m", "uv"]
        if not PY.exists():
            stream([*uv, "venv", "--python", sys.executable, PY.parent.parent])
        install = [*uv, "pip", "install", "--python", PY, "--torch-backend=auto"]
        stream([*install, TORCH])  # a few GB of CUDA libraries the first time
        stream([*install, TORCH, *PACKAGES])  # torch named again, so nothing replaces the build just chosen
        stream([*install, "--no-deps", "--reinstall", *FORKS])
elif ENVIRONMENT == "kernel":
    PY = Path(sys.executable)
    if INSTALL:
        worked, torch_version, _ = in_python("import torch; print(torch.__version__)")
        if not worked:
            raise RuntimeError("torch is not importable in this kernel: install it first, or use ENVIRONMENT = 'own'")
        # A torch built against NumPy 1.x cannot use NumPy 2, and pyarrow 26 needs NumPy 2 without declaring it, so
        # pip sees neither conflict. Where NumPy is 1.x, or torch cannot read the NumPy 2 installed, stay on 1.x.
        _, numpy_version, _ = in_python("import importlib.metadata as m; print(m.version('numpy'))")
        reads_numpy, _, _ = in_python("import numpy, torch; torch.from_numpy(numpy.zeros(1))")
        keep_numpy_1 = ["numpy<2", "pyarrow>=24,<26"] if numpy_version.startswith("1.") or not reads_numpy else []
        if keep_numpy_1:
            print(f"torch {torch_version} with NumPy {numpy_version or 'missing'}: keeping NumPy 1.x for this torch")
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        stream([*pip, *PACKAGES, *keep_numpy_1])
        stream([*pip, "--no-deps", "--force-reinstall", *FORKS])
else:
    raise ValueError("ENVIRONMENT must be 'own' or 'kernel'")

check = (
    "import accelerate, datasets, numpy, peft, torch, transformers, trl; torch.from_numpy(numpy.zeros(1)); "
    "from transformers import ClefForDecision, ClmForDecision, JevaForDecision, JevForDecision; "
    "from transformers.loss.loss_utils import LOSS_MAPPING; "
    "assert 'ClefForDecision' in LOSS_MAPPING, 'this transformers has no Clef training support: is the fork installed?'; "
    "from trl.experimental.decision import DecisionTrainer, target_from_gold; "
    "print('torch', torch.__version__, '| numpy', numpy.__version__, '| transformers', transformers.__version__, "
    "'| trl', trl.__version__, '| accelerate', accelerate.__version__, '| peft', peft.__version__)"
)
worked, versions, errors = in_python(check)
print(versions if worked else errors)
if not worked:
    raise RuntimeError("This environment cannot import what the notebook needs; see the message above")

# The example scripts are fetched at the pinned TRL commit, so the environment has to hold the same code
where_from = "import importlib.metadata as m, json; print(json.loads(m.distribution({!r}).read_text('direct_url.json') or '{{}}').get('url', ''))"
for package, source in (("transformers", TRANSFORMERS_SOURCE), ("trl", TRL_SOURCE)):
    _, installed, _ = in_python(where_from.format(package))
    if installed != source:
        raise RuntimeError(
            f"The environment's {package} was installed from {installed or 'a release or a local checkout'},\n"
            f"and this notebook is written for {source}.\nSet INSTALL = True and run this cell again."
        )
print("Python of the runs:", PY)

## The example scripts and this machine

The four examples and their configuration files are fetched from the TRL fork at the pinned commit. The GPUs are
counted in a separate process. The flash-attention kernel of the Clef stage is tried once here: where it has no build
for the machine's torch, the Clef backbone uses the default attention instead, and the cell says so.

In [ ]:
STAGES = {"clm": "CLM", "jeva": "Jeva", "jev": "JEV", "clef": "Clef"}
unknown = [stage for stage in RUN if stage not in STAGES]
if unknown:
    raise ValueError(f"RUN names {unknown}; the stages are {list(STAGES)}")

import io
import re
import tarfile
import urllib.request

# One folder per pinned TRL commit: a run folder that an earlier version of this notebook used never supplies its
# older scripts. A source that names no commit (a branch) may have changed, so it is fetched every time.
pinned = re.search(r"[0-9a-f]{40}", TRL_SOURCE)
EXAMPLES = WORK_DIR / "examples" / (pinned.group()[:12] if pinned else "unpinned")
SCRIPTS = {
    "clm": EXAMPLES / "examples/decisions/clm_typed_decisions.py",
    "jeva": EXAMPLES / "examples/decisions/jeva_banking77.py",
    "jev": EXAMPLES / "examples/decisions/jev.py",
    "clef": EXAMPLES / "examples/decisions/clef_typed_decisions.py",
}
if pinned is None or not all(script.exists() for script in SCRIPTS.values()):
    with urllib.request.urlopen(TRL_SOURCE) as response:
        archive = tarfile.open(fileobj=io.BytesIO(response.read()), mode="r:gz")
    for member in archive.getmembers():
        parts = Path(member.name).parts[1:]  # without the top folder of the archive
        if member.isfile() and parts[:2] == ("examples", "decisions"):
            target = EXAMPLES.joinpath(*parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.extractfile(member).read())

probe = (
    "import json, torch; n = torch.cuda.device_count() if torch.cuda.is_available() else 0; "
    "print(json.dumps({'gpus': n, 'names': [torch.cuda.get_device_name(i) for i in range(n)], 'torch': torch.__version__}))"
)
worked, found, errors = in_python(probe)
if not worked:
    raise RuntimeError(errors)
machine = json.loads(found.splitlines()[-1])
ON_GPU = machine["gpus"] > 0
if NUM_GPUS is not None and NUM_GPUS > machine["gpus"]:
    raise ValueError(f"NUM_GPUS is {NUM_GPUS}, and this machine shows {machine['gpus']}")
PROCESSES = (NUM_GPUS or machine["gpus"]) if ON_GPU else 1

ATTENTION = CLEF_ATTENTION if ON_GPU and "clef" in RUN else None
if ATTENTION is not None and "/" in ATTENTION:  # a kernel from the Hub: does it have a build for this torch?
    trial = f"from transformers.modeling_flash_attention_utils import lazy_import_flash_attention as load; load({ATTENTION!r})"
    worked, _, errors = in_python(trial)
    if not worked:
        reason = next((line for line in reversed(errors.splitlines()) if line.strip()), "no reason given")
        print(f"The flash-attention kernel does not load here, so the backbone uses the default attention.\n  {reason[:300]}")
        ATTENTION = None

print("torch", machine["torch"], "|", machine["names"] or "no CUDA GPU: CLM and Clef run a few steps on the CPU, only to check them")
print(PROCESSES, "process(es) per training run | example scripts:", EXAMPLES)
if "clef" in RUN:
    print("Clef backbone attention:", ATTENTION or "default")

## How each stage runs

A stage starts from an empty output folder, so nothing of an earlier run is read. Its training command is printed on
the line that starts with `$`, and so is the command that asks the trained model: both work as they are in a terminal,
from the run folder. The small scripts this notebook writes (asking a model, building JEV's requests) are in
`helpers/`.

In [ ]:
HELPERS = WORK_DIR / "helpers"  # a folder of their own, so nothing else in the run folder is imported with them
HELPERS.mkdir(exist_ok=True)
SUMMARY = {}  # what happened to each stage, for the table at the end


def launch(script, *arguments):
    command = [PY, "-m", "accelerate.commands.launch", "--num_processes", PROCESSES, "--num_machines", 1]
    command += ["--mixed_precision", "bf16" if ON_GPU else "no", "--dynamo_backend", "no"]
    if PROCESSES > 1:
        command += ["--multi_gpu", "--main_process_port", 0]
    if not ON_GPU:
        command += ["--cpu"]
    stream([*command, script, *arguments])


def arguments(settings):
    """The command-line form of `settings`: `--name value`, with booleans as true / false."""
    line = []
    for name, value in settings.items():
        line += [f"--{name}", str(value).lower() if isinstance(value, bool) else value]
    return line


def fresh(folder):
    """Empty a folder of the run folder before a stage writes to it."""
    if folder.exists():
        shutil.rmtree(folder)


def left_out(stage):
    print(f"{STAGES[stage]} is left out: it is not in RUN.")
    SUMMARY[stage] = {"outcome": "left out", "note": "not in RUN"}


def skipped(stage, reason):
    print(f"{STAGES[stage]} is skipped: {reason}.")
    SUMMARY[stage] = {"outcome": "skipped", "note": reason}


def report(stage, output, before=None):
    """Print the final evaluation the example wrote (and the one before training, where it writes one)."""
    runs = {"after": json.loads((output / "eval_results.json").read_text())}
    if before is not None:
        runs = {"before": json.loads((output / f"{before}_results.json").read_text()), **runs}
    names = [key for key, value in runs["after"].items() if key.startswith("eval_") and isinstance(value, (int, float))]
    names = [name for name in names if not name.endswith(("_runtime", "_per_second", "_preparation_time"))]
    print(f"{'':24}" + "".join(f"{when:>12}" for when in runs))
    for name in names:
        print(f"{name[5:]:24}" + "".join(f"{metrics[name]:12.4f}" if name in metrics else f"{'':>12}" for metrics in runs.values()))
    headline = "eval_accuracy" if "eval_accuracy" in runs["after"] else "eval_loss"
    SUMMARY[stage] = {
        "outcome": "ran",
        "metric": headline[5:],
        "before": runs.get("before", {}).get(headline),
        "after": runs["after"].get(headline),
        "output": output,
    }
    if QUICK:
        print("QUICK run: these numbers only show that the stage works.")


ASK_SCRIPT = HELPERS / "ask.py"
ASK_SCRIPT.write_text('''
"""Ask a trained decision model one request through pipeline("decision") and print its answer.

python ask.py --model OUTPUT_FOLDER --request REQUEST.json [--adapter] [--without_saved_adapter] [--float32]
"""
import argparse
import dataclasses
import json
from pathlib import Path

import torch
from transformers import AutoModelForDecision, AutoTokenizer, pipeline

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True, help="the output folder of a training run")
parser.add_argument("--request", required=True, help="a JSON file with a state and its questions")
parser.add_argument("--adapter", action="store_true", help="the folder holds a LoRA adapter, its base in base_model/")
parser.add_argument("--without_saved_adapter", action="store_true", help="ignore the adapter the decision config names")
parser.add_argument("--float32", action="store_true", help="float32 weights on a GPU too (under bf16 autocast)")
args = parser.parse_args()

on_cuda = torch.cuda.is_available()
dtype = torch.float32 if args.float32 or not on_cuda else torch.bfloat16
device = 0 if on_cuda else -1
if args.adapter:
    from peft import PeftModel

    base = AutoModelForDecision.from_pretrained(Path(args.model) / "base_model", dtype=dtype)
    model = PeftModel.from_pretrained(base, args.model).merge_and_unload()
    decide = pipeline("decision", model=model, tokenizer=AutoTokenizer.from_pretrained(args.model), device=device)
elif args.without_saved_adapter:
    from transformers.decision import DecisionConfig

    config = dataclasses.replace(DecisionConfig.from_pretrained(args.model), adapter=None)
    decide = pipeline("decision", model=args.model, decision_config=config, dtype=dtype, device=device)
else:
    decide = pipeline("decision", model=args.model, dtype=dtype, device=device)
request = json.loads(Path(args.request).read_text())
with torch.autocast("cuda", dtype=torch.bfloat16, enabled=on_cuda):
    answer = decide(request)
print(json.dumps(answer, indent=2, default=str))
'''.lstrip())


def ask(stage, output, request, *options):
    """Ask the model saved in `output` one request, in a fresh process of the environment; `options` of ask.py."""
    request_file = HELPERS / f"{stage}_request.json"
    request_file.write_text(json.dumps(request, indent=2) + "\n")
    print("request:", json.dumps(request))
    stream([PY, ASK_SCRIPT, "--model", output, "--request", request_file, *options])


# A support ticket with the three kinds of question: noul (yes or no), choice and score
TICKET = {
    "state": {"ticket": "My card was charged twice for the same order and I want one charge back today."},
    "questions": {
        "refund": {"type": "noul", "instructions": "Is the customer asking for a refund?"},
        "team": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {"billing": "Charges and refunds", "shipping": "Delivery", "tech": "App problems"},
        },
        "urgency": {"type": "score", "instructions": "How urgent is it?", "criteria": ["Can wait", "This week", "Today"]},
    },
}

## CLM: projection heads on a frozen encoder

`clm_typed_decisions.py` keeps a SmolLM2-135M encoder frozen and trains small projection heads on it, so that the
embedding of a question's state lands close to the embedding of its right option. It trains on the typed-decisions
dataset, one question per row, and evaluates before and after training. A 135M encoder is small enough for a CPU, so
without a GPU this stage runs a few steps there.

In [ ]:
CLM_OUTPUT = WORK_DIR / "clm"
if "clm" not in RUN:
    left_out("clm")
else:
    settings = {"model_name_or_path": CLM_ENCODER, "dataset_name": DATASET, "dataset_config": DATASET_CONFIG}
    settings.update(output_dir=CLM_OUTPUT, seed=SEED, disable_tqdm=True)
    if QUICK and ON_GPU:
        settings.update(num_train_epochs=2, max_train_samples=200, max_eval_samples=40)
    elif QUICK:
        settings.update(num_train_epochs=1, max_train_samples=24, max_eval_samples=8, max_length=512)
    if not ON_GPU:
        settings.update(use_cpu=True, bf16=False)
    fresh(CLM_OUTPUT)
    launch(SCRIPTS["clm"], "--config", SCRIPTS["clm"].with_suffix(".yaml"), *arguments(settings))
    report("clm", CLM_OUTPUT, before="eval_before")
    ask("clm", CLM_OUTPUT, TICKET)

## Jeva: a decision head with LoRA on Qwen3.5-4B

`jeva_banking77.py` trains a Jeva decision head, with LoRA adapters on a Qwen3.5-4B encoder, on BANKING77: each
request asks which of eight banking intents matches a customer's message, and in some of them none does. The example
saves the adapter in its output folder and the base model it goes on in `base_model/`, so the trained model is asked
by loading that base, putting the adapter on it and merging the two. A 4B encoder needs a GPU.

In [ ]:
JEVA_OUTPUT = WORK_DIR / "jeva"
if "jeva" not in RUN:
    left_out("jeva")
elif not ON_GPU:
    skipped("jeva", f"there is no CUDA GPU here, and its encoder ({JEVA_ENCODER}, 4B parameters) does not train on a CPU in a short run")
else:
    settings = {"model_name_or_path": JEVA_ENCODER, "output_dir": JEVA_OUTPUT, "seed": SEED, "disable_tqdm": True}
    if QUICK:
        settings.update(max_steps=40, eval_steps=20, save_steps=20, max_train_samples=400, max_eval_samples=64, logging_steps=5)
    fresh(JEVA_OUTPUT)
    launch(SCRIPTS["jeva"], "--config", SCRIPTS["jeva"].with_suffix(".yaml"), *arguments(settings))
    report("jeva", JEVA_OUTPUT)
    ask("jeva", JEVA_OUTPUT, {
        "state": "I still have not received my new card, it has been two weeks.",
        "questions": {
            "intent": {
                "type": "choice",
                "instructions": "Which banking intent best matches this request?",
                "criteria": {"card arrival": None, "lost or stolen card": None, "card payment fee charged": None, "exchange rate": None},
            }
        },
    }, "--adapter")

## JEV: a full fine-tune of JEV-9B

`jev.py` fine-tunes every weight of a JEV checkpoint (`JEV_MODEL`) on rows that hold one question each, with the
probability of each of its options as the target. This stage first builds those rows from the typed-decisions
dataset: one row per question, with the whole of a source record kept in one split. Training a 9B model in full at
batch size one needs a lot of memory, so the stage measures what is available to the GPU first and is skipped below
`JEV_MIN_FREE_GB`. On a DGX Spark the quick run with `JEV_OPTIM = "sgd"` took about 53 GB (measured); the example's
default AdamW keeps two more copies of the model, about 90 GB in all (an estimate, not run), so with AdamW raise
`JEV_MIN_FREE_GB` to about 95.

The example fine-tunes the checkpoint's base weights and head, without the LoRA that JEV-9B keeps in `adapter/`, and
the saved decision config still names that adapter, which the output folder does not have: so the trained model is
asked with the decision config without it (`--without_saved_adapter`).

In [ ]:
JEV_SCRIPT = HELPERS / "jev_requests.py"
JEV_SCRIPT.write_text('''
"""Rows for jev.py built from a typed-decisions dataset: one request per question, whole source rows in one split.

python jev_requests.py --model JEV_MODEL --output FOLDER [--train 200 --test 40]

A typed-decisions row holds a state, several questions and the gold answer of each. jev.py trains on rows that hold
one question each: a `request` (the state and one question) and its `target_probs` (null first, then one per option).
Questions the model cannot take (nullable ones, more options than its head scores) are left out and counted.
"""
import argparse
import json
import random
from collections import Counter
from pathlib import Path

from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, JevForDecision

from trl.experimental.decision import DataCollatorForDecision, target_from_gold

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True)
parser.add_argument("--dataset", default="LocalLLaMA/typed-decisions")
parser.add_argument("--dataset_config", default="all")
parser.add_argument("--output", required=True)
parser.add_argument("--train", type=int, default=None, help="training requests (default: all)")
parser.add_argument("--test", type=int, default=None, help="evaluation requests (default: all)")
parser.add_argument("--seed", type=int, default=0)
args = parser.parse_args()

# The decision config (template, option limits) is read from the checkpoint's files, without its weights
_, decision_config = JevForDecision.checkpoint_config(args.model)
if decision_config is None:
    raise SystemExit(f"{args.model} is not a JEV checkpoint")
collator = DataCollatorForDecision(AutoTokenizer.from_pretrained(args.model), decision_config)
rows = list(load_dataset(args.dataset, args.dataset_config, split="train"))
random.Random(args.seed).shuffle(rows)
held_out = max(1, len(rows) // 10)  # whole rows go to one split, so no state is in both
output, left_out = Path(args.output), Counter()
output.mkdir(parents=True, exist_ok=True)
for split, source, count in (("train", rows[held_out:], args.train), ("test", rows[:held_out], args.test)):
    requests = []
    for row in source:
        state, questions, gold = (json.loads(row[key]) for key in ("state", "questions", "gold"))
        for name, question in questions.items():
            request = {"state": state, "questions": {name: question}}
            try:
                _, parsed, _ = collator.prepare_example({"request": request, "target_index": 0})
                target = target_from_gold(gold[name], parsed)
                collator.prepare_example({"request": request, **target})
            except (ValueError, KeyError) as error:
                left_out[str(error).split(":")[0][:60]] += 1
                continue
            requests.append({"request": json.dumps(request), **target})
        if count is not None and len(requests) >= count:
            break
    requests = requests if count is None else requests[:count]
    Dataset.from_list(requests).to_parquet(output / f"{split}.parquet")
    print(f"{split}: {len(requests)} requests")
print("questions left out:", dict(left_out) or "none")
'''.lstrip())

JEV_OUTPUT, JEV_REQUESTS = WORK_DIR / "jev", WORK_DIR / "jev_requests"
if "jev" not in RUN:
    left_out("jev")
elif not ON_GPU:
    skipped("jev", "there is no CUDA GPU here, and it fine-tunes every weight of a 9B model")
else:
    # The memory the GPU can take, measured in a fresh process. A GPU that shares the machine's memory (a DGX Spark)
    # can take what the system has available, page cache included, which CUDA's own figure leaves out.
    probe = (
        "import json, torch; from pathlib import Path; "
        f"free = min(torch.cuda.mem_get_info(i)[0] for i in range({PROCESSES})) / 2**30; "
        "shared = bool(getattr(torch.cuda.get_device_properties(0), 'is_integrated', False)); "
        "meminfo = Path('/proc/meminfo'); "
        "system = next((int(line.split()[1]) / 2**20 for line in meminfo.read_text().splitlines() if line.startswith('MemAvailable:')), None) if meminfo.exists() else None; "
        "print(json.dumps({'cuda_free': free, 'shared': shared, 'system_available': system}))"
    )
    worked, found, errors = in_python(probe)
    if not worked:
        raise RuntimeError(errors)
    memory = json.loads(found.splitlines()[-1])
    available = memory["system_available"] if memory["shared"] and memory["system_available"] is not None else memory["cuda_free"]
    print(f"memory available to the GPU: {available:.0f} GB (CUDA reports {memory['cuda_free']:.0f} GB free"
          + (f"; the GPU shares the machine's memory, of which {memory['system_available']:.0f} GB are available)" if memory["shared"] else ")"))
    if available < JEV_MIN_FREE_GB:
        skipped("jev", f"{available:.0f} GB are available to the GPU, and JEV_MIN_FREE_GB asks for {JEV_MIN_FREE_GB} GB")
    else:
        fresh(JEV_REQUESTS)
        sizes = ["--train", 200, "--test", 40] if QUICK else []
        stream([PY, JEV_SCRIPT, "--model", JEV_MODEL, "--dataset", DATASET, "--dataset_config", DATASET_CONFIG, "--output", JEV_REQUESTS, "--seed", SEED, *sizes])
        settings = {"model_name_or_path": JEV_MODEL, "dataset_name": JEV_REQUESTS, "output_dir": JEV_OUTPUT}
        settings.update(per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_checkpointing=False)
        settings.update(dtype="bfloat16", bf16=True, optim=JEV_OPTIM, gradient_accumulation_steps=4, learning_rate=1e-5)
        settings.update(eval_strategy="steps", save_strategy="no", report_to="none", seed=SEED, disable_tqdm=True)
        if QUICK:
            settings.update(max_steps=40, eval_steps=20, logging_steps=5)
        else:
            settings.update(num_train_epochs=1, eval_steps=200, logging_steps=10)
        fresh(JEV_OUTPUT)
        launch(SCRIPTS["jev"], *arguments(settings))
        report("jev", JEV_OUTPUT)
        ask("jev", JEV_OUTPUT, TICKET, "--without_saved_adapter")

## Clef: a cosine head with its KLM-1.6 backbone

`clef_typed_decisions.py` trains a small cosine head together with its KLM-1.6 backbone on the typed-decisions
dataset; one prompt answers every question of a record. This stage is the cross-entropy run of
`clef_decision_training.ipynb`, with the same configuration and the same quick sizes. That notebook goes on from
here: the RLCD phase that continues training after cross-entropy, the converter for older checkpoints, and the
benchmark.

In [ ]:
CLEF_OUTPUT = WORK_DIR / "clef"
CLEF_BATCH = 32  # records per optimizer step, split evenly across the GPUs
if "clef" not in RUN:
    left_out("clef")
else:
    if ON_GPU and CLEF_BATCH % PROCESSES:
        raise ValueError(f"The Clef batch of {CLEF_BATCH} records does not divide evenly across {PROCESSES} GPUs")
    settings = {"model_name_or_path": CLEF_MODEL, "model_revision": CLEF_MODEL_REVISION}
    settings.update(dataset_name=DATASET, dataset_config=DATASET_CONFIG, output_dir=CLEF_OUTPUT)
    settings.update(
        per_device_train_batch_size=CLEF_BATCH // PROCESSES if ON_GPU else 2,
        per_device_eval_batch_size=8 if ON_GPU else 2,
        optim="adamw_torch_fused" if ON_GPU else "adamw_torch",
        weight_decay=0.05,
        warmup_steps=0.03,
        logging_steps=12,
        activation_checkpointing=ON_GPU,
        bf16=ON_GPU,
        dataloader_num_workers=4 if ON_GPU else 0,
        disable_tqdm=True,
        seed=SEED,
    )
    if ATTENTION is not None:
        settings["attn_implementation"] = ATTENTION
    if not ON_GPU:
        settings["use_cpu"] = True
    if QUICK and ON_GPU:  # 20 steps per epoch, 2 epochs
        settings.update(max_train_samples=CLEF_BATCH * 20, max_eval_samples=128, num_train_epochs=2, eval_steps=10, logging_steps=10)
    elif QUICK:  # a handful of steps on a CPU
        settings.update(max_train_samples=16, max_eval_samples=8, max_steps=4, eval_steps=2, logging_steps=1)
    fresh(CLEF_OUTPUT)
    launch(SCRIPTS["clef"], "--config", SCRIPTS["clef"].with_suffix(".yaml"), *arguments(settings))
    report("clef", CLEF_OUTPUT)
    ask("clef", CLEF_OUTPUT, TICKET, "--float32")  # float32 weights under bf16 autocast, as KLM's kernels need

## Summary

One line per stage: whether it ran, was left out or was skipped, its headline metric on the held-out records (before
and after training where the example evaluates both), and where its output is.

In [ ]:
rows = [("stage", "outcome", "metric", "before", "after", "output", "why")]
for stage, name in STAGES.items():
    done = SUMMARY.get(stage, {"outcome": "not run", "note": "its cell was not executed"})
    number = lambda value: "" if value is None else f"{value:.4f}"
    output = done.get("output")
    output = str(output.relative_to(WORK_DIR)) + "/" if output else ""
    rows.append((name, done["outcome"], done.get("metric", ""), number(done.get("before")), number(done.get("after")), output, done.get("note", "")))
widths = [max(len(row[column]) for row in rows) for column in range(len(rows[0]))]
for row in rows:
    print("  ".join(cell.ljust(width) for cell, width in zip(row, widths)).rstrip())
print("\nrun folder:", WORK_DIR)
if QUICK:
    print("QUICK run: the numbers only show that the stages work, not how well the examples train.")

## Notes

**What the quick sizes are for.** `QUICK = True` trains each example for a few dozen steps on a sample (on a CPU, a
handful of steps on a few records), enough to see it train, evaluate, save and answer. Its numbers say nothing about
how well an example trains. `QUICK = False` runs each example's own recipe: the full configuration file of CLM, Jeva
and Clef, and one epoch over every request for JEV.

**The run folder.** `decision_examples_run/` can be used again, by this notebook or a later version of it. The
environment in `env/` is checked against the pinned sources (set `INSTALL = False` to use it as it is, or delete it to
start again), the example scripts are kept per pinned commit under `examples/`, and each stage empties its own output
folder before it trains.

**In a terminal.** Each command of a stage is printed on the line that starts with `$` and works as it is in a
terminal, from the run folder. For one GPU, the CLM stage looks like this (with several, `--multi_gpu` and one process
per GPU):

```bash
env/bin/python -m accelerate.commands.launch --num_processes 1 --num_machines 1 --mixed_precision bf16 --dynamo_backend no \
    examples/<commit>/examples/decisions/clm_typed_decisions.py --config examples/<commit>/examples/decisions/clm_typed_decisions.yaml \
    --model_name_or_path HuggingFaceTB/SmolLM2-135M --output_dir clm --num_train_epochs 2 --max_train_samples 200 --max_eval_samples 40
env/bin/python helpers/ask.py --model clm --request helpers/clm_request.json
```

**Clef in depth.** `clef_decision_training.ipynb` trains Clef with cross-entropy as here, then continues with the
RLCD phase, shows the metrics of both runs, and has the converter and the benchmark.